In [1]:
from parameter.aparameter import AParameter
from strategy.strategy_factory import StrategyFactory
from transformer.transformer import Transformer
from backtester.backtester import Backtester
from processor.server_processor import ServerProcessor
from datetime import datetime, timedelta
from strategy.strategy import Strategy
import pandas as pd
from tqdm import tqdm
from database.adatabase import ADatabase
import matplotlib.pyplot as plt

In [2]:
db = ADatabase("sapling")

In [3]:
db.connect()
kpi = db.retrieve("kpi")
db.disconnect()

In [4]:
strategy_kpis = kpi.sort_values("return",ascending=False).groupby(["strategy"]).first().sort_values("return",ascending=False).reset_index()

In [5]:
strategy_kpis

,strategy,number_of_trades,standard_deviation,coefficient_of_variance,sharpe,return,holding_period,positions,stop_loss,ascending
0,KURTOSIS,102,71.1994,0.7111,3.0169,214.8038,5,1,1,False
1,STANDARD_DEVIATION,102,83.2023,1.7389,2.3140,192.5313,5,1,1,True
2,UNITY_AI,81,50.2038,0.7115,3.2527,163.2969,5,1,1,True
3,RSI,510,74.5959,0.7355,2.0407,152.2278,5,5,1,True
4,STOCHASTIC_OSCILLATOR,1020,57.4421,0.7261,2.4035,138.0593,5,10,1,False
5,RANDOM,102,25.7060,0.2535,5.3668,137.9601,5,1,1,True
6,BOLLINGER_WIDTH,102,53.9373,1.5699,2.5077,135.2579,5,1,1,True
7,COEFFICIENT_OF_VARIANCE,102,53.9373,1.5699,2.5077,135.2579,5,1,1,True
8,BOLLINGER,102,123.6026,0.7425,1.0505,129.8493,5,1,1,False
9,MACD,1020,40.4493,0.6308,2.6706,108.0251,5,10,1,True


In [6]:
db = ADatabase("sapling")
market = ADatabase("market")
market.connect()
tickers = market.retrieve("russell1000")["ticker"].values
market.disconnect()
strategy_kpis["tickers"] = [tickers for i in range(strategy_kpis.index.size)]
query = strategy_kpis.to_dict("records")[0]

In [7]:
analysis = []
db.connect()
recs = []
try:
    start = datetime.now() - timedelta(days=365.25)
    end = datetime.now() - timedelta(hours=24)
    parameter = AParameter()
    parameter.build(query)
    strategy = StrategyFactory.build(parameter)
    simulation = Transformer.local_transform(strategy,start,end)
    trades = Backtester.backtest(strategy,simulation)
    portfolio = Backtester.portfolio(trades)
    recommendations = Backtester.recommendations(trades)
    kpi = Backtester.kpi(trades,portfolio)
    results = ServerProcessor.server_format(strategy,trades,portfolio,recommendations,kpi)
    recs.extend(results["recommendations"])
except Exception as e:
    print(str(e))
db.disconnect()

In [8]:
recommendations = pd.DataFrame(recs)

In [9]:
recommendations

,date,buy_date,sell_date,ticker,adjclose,kurtosis
0,2024-02-09,2024-02-12,2024-02-16,SNA,262.43,8.0951


In [10]:
db.connect()
db.drop("recommendations")
db.store("recommendations",recommendations)
db.disconnect()